In [ ]:
!pip install peft bitsandbytes trl accelerate

In [ ]:
from huggingface_hub import login

# Run this and paste your token when prompted
login()

In [ ]:
import torch
from datasets import load_dataset, concatenate_datasets
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig # Import SFTConfig

# 1. Configuration
# ----------------
model_name = "meta-llama/Meta-Llama-3-8B-Instruct"
new_model_name = "Llama-3-Engineering-Finetune"

# QLoRA config for 4-bit quantization (T4 GPU compatible)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

# LoRA Configuration
peft_config = LoraConfig(
    r=16,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=['k_proj', 'q_proj', 'v_proj', 'o_proj', "gate_proj", "down_proj", "up_proj"]
)

# 2. Load Data
# ------------
# Load both JSONL files
dataset_nuts = load_dataset("json", data_files="nuts_and_bolts_manifest.jsonl", split="train")
dataset_gears = load_dataset("json", data_files="gear_training_manifest.jsonl", split="train")
full_dataset = concatenate_datasets([dataset_nuts, dataset_gears]).shuffle(seed=42)
#full_dataset = (dataset_nuts).shuffle(seed=42)

print(f"Total training examples: {len(full_dataset)}")
print("Example entry:", full_dataset[0])


# 3. Formatting Function (To create a single text sequence)
# --------------------------------------------------------
def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    outputs = examples["output"]
    texts = []
    for instruction, output in zip(instructions, outputs):
        # Apply the standard Llama 3 Instruction Fine-Tuning format (Alpaca style)
        text = f"""Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
{instruction}

### Response:
{output}<|eot_id|>"""
        texts.append(text)
    return texts

# FIX for 'list' AttributeError: Apply formatting manually
print("Applying formatting to create 'text' column...")
def apply_formatting_to_batch(batch):
    # This creates the necessary single 'text' string from 'instruction' and 'output'
    return {"text": formatting_prompts_func(batch)}

formatted_dataset = full_dataset.map(apply_formatting_to_batch, batched=True, remove_columns=["instruction", "output"])
print("Formatted dataset column names:", formatted_dataset.column_names)


# 4. Load Model and Tokenizer
# ---------------------------
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    use_cache=False
)
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# 5. Training Arguments (using SFTConfig) - FINAL MEMORY FIX
# ---------------------------------------
training_arguments = SFTConfig(
    output_dir="./results",
    num_train_epochs=1,
    max_steps=100,
    # REDUCED: Batch size from 4 to 2 (to save VRAM)
    per_device_train_batch_size=2,
    # INCREASED: Accumulation steps to compensate for smaller batch size:
    # Effective batch size remains 4 (2 * 2 = 4)
    gradient_accumulation_steps=2,
    optim="paged_adamw_32bit",
    save_steps=25,
    logging_steps=25,
    learning_rate=2e-4,
    weight_decay=0.001,
    lr_scheduler_type="constant",

    # REQUIRED FOR STABILITY
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},

    # REDUCED: Max sequence length from 1024 to 600 (to save VRAM)
    max_length=600,
    packing=False,
    dataset_text_field="text",
)
# 6. Initialize Trainer
# ---------------------
trainer = SFTTrainer(
    model=model,
    train_dataset=formatted_dataset,
    peft_config=peft_config,
    # FIXED: Use 'processing_class' instead of 'tokenizer'
    processing_class=tokenizer,
    args=training_arguments,
)

# 7. Start Training
# -----------------
print("\nStarting fine-tuning...")
trainer.train()

# 8. Save the Model
# -----------------
trainer.model.save_pretrained(new_model_name)
tokenizer.save_pretrained(new_model_name)
print(f"\nModel adapters saved to {new_model_name}")

Generating train split: 0 examples [00:00, ? examples/s]

Generating train split: 0 examples [00:00, ? examples/s]

Total training examples: 24199
Example entry: {'instruction': 'Design a gear that satisfies the following requirements: module mm = 4, number of teeth pinion = 17, and number of teeth gear = 50. Provide the complete gear parameters (module, teeth counts, diameters, face width, addendum/dedendum, pressure angle, center distance, lubrication) as a JSON gear_design object.', 'output': '{"gear_design":{"gear_type":"bevel","module_mm":4.0,"number_of_teeth_pinion":17,"number_of_teeth_gear":50,"pitch_diameter_pinion_mm":68.0,"pitch_diameter_gear_mm":200.0,"face_width_mm":39.249,"addendum_mm":4.0,"dedendum_mm":5.0,"pressure_angle_deg":20,"center_distance_mm":134.0,"input_torque_Nm":4235.8683,"input_speed_rpm":2999,"input_power_kW":1330.2937,"helix_angle_deg":null,"recommended_face_width_mm":44.983,"module_candidates":[4.0],"lubrication_recommendation":"ISO VG 460 (heavy)"}}'}
Applying formatting to create 'text' column...


Map:   0%|          | 0/24199 [00:00<?, ? examples/s]

Formatted dataset column names: ['text']


...

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: aman_k1 (aman_k1-iit-roorkee) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step,Training Loss
25,0.784400
50,0.334700
75,0.315000
100,0.309700



Model adapters saved to Llama-3-Engineering-Finetune


In [ ]:
wand_ai_api_key = "a137a6fa3c3d78f41abb383b20b37c54feb77b0d"

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
import json

# --- Configuration (must match training) ---
model_name = "meta-llama/Meta-Llama-3-8B-Instruct"
new_model_name = "Llama-3-Engineering-Finetune"

# --- Load Base Model and Adapters ---

# 1. Load the base model in 4-bit (just like during training)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
)

# 2. Load the tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

# 3. Load the LoRA adapters onto the base model
model = PeftModel.from_pretrained(base_model, new_model_name)

# 4. Optional: Merge the LoRA adapters into the base model weights
# This is required if you want to save the final model for easy deployment.
# We skip merging here to keep it simple, but you can add it if needed:
# model = model.merge_and_unload()
print("Model loaded and adapters applied. Ready for inference! 🚀")

# --- Test Inference ---

# Define a test instruction (e.g., a query similar to your training data)
instruction = "Design a bolt that satisfies the following requirements: size = M10 and grade = 10.9. Return only the bolt_spec as JSON."

# Format the prompt using the exact template used during fine-tuning
prompt = f"""Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
{instruction}

### Response:
"""

# Tokenize and move to GPU
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

# Generate the output
outputs = model.generate(
    **inputs,
    max_new_tokens=256,
    eos_token_id=tokenizer.eos_token_id,
    do_sample=True,             # Use sampling to get varied/creative responses
    temperature=0.7,
    top_p=0.9
)

# Decode the response
response = tokenizer.decode(outputs[0], skip_special_tokens=True)

# --- Display Results ---
print("\n" + "="*50)
print("⚙️ TEST PROMPT ⚙️")
print(prompt)
print("="*50)
print("🤖 GENERATED RESPONSE 🤖")

# Attempt to extract the JSON output only
try:
    # Find the '### Response:' part and clean up the end of sequence token
    response_text = response.split("### Response:")[1].strip()
    # Attempt to parse as JSON (optional, but good for verification)
    parsed_json = json.loads(response_text.replace("<|eot_id|>", "").strip())

    print("✅ Parsed JSON Output:")
    print(json.dumps(parsed_json, indent=4))

except Exception as e:
    print("⚠️ Raw Output (JSON parsing failed):")
    print(response_text)
    print(f"Error: {e}")

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:128009 for open-end generation.


Model loaded and adapters applied. Ready for inference! 🚀

⚙️ TEST PROMPT ⚙️
Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
Design a bolt that satisfies the following requirements: size = M10 and grade = 10.9. Return only the bolt_spec as JSON.

### Response:

🤖 GENERATED RESPONSE 🤖
✅ Parsed JSON Output:
{
    "bolt_spec": {
        "size": "M10",
        "grade": "10.9",
        "pitch_mm": 1.5,
        "stress_area_mm2": 58.0
    }
}


In [ ]:
from google.colab import files
import shutil

# The name of the folder containing your LoRA adapters
model_folder_name = "Llama-3-Engineering-Finetune"

# Create a zip archive of the directory
print(f"Compressing folder: {model_folder_name}...")
shutil.make_archive(model_folder_name, 'zip', model_folder_name)

# Trigger the download in your browser
print("Starting download... Check your browser downloads.")
files.download(f"{model_folder_name}.zip")

Compressing folder: Llama-3-Engineering-Finetune...
Starting download... Check your browser downloads.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from huggingface_hub import HfApi

# --- Configuration ---
local_model_directory = "Llama-3-Engineering-Finetune"

# *** FIX THIS LINE: REPLACE 'YourUsername' with your actual Hugging Face username ***
# Example: If your username is 'engineering_guru', it should be: "engineering_guru/Llama-3-Engineering-Finetune-QLoRA"
repo_id = "Black4cosmos/Llama-3-Engineering-Finetune-QLoRA"

# Initialize the Hugging Face API
api = HfApi()

print("--- Starting Upload Process ---")

# 1. Create the repository on the Hugging Face Hub
api.create_repo(
    repo_id=repo_id,
    exist_ok=True,
    repo_type="model",
    private=False # Set to True if you want the model to be private
)

print(f"Created/verified repository: {repo_id}")

# 2. Upload all files from your local Colab directory to the remote repo
api.upload_folder(
    folder_path=local_model_directory,
    repo_id=repo_id,
    repo_type="model"
)

print("\n✅ Success! Your LoRA adapters have been pushed to the Hugging Face Hub.")
print(f"You can view and use your model from this link:")
print(f"https://huggingface.co/{repo_id}")

--- Starting Upload Process ---
Created/verified repository: Black4cosmos/Llama-3-Engineering-Finetune-QLoRA


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...g-Finetune/tokenizer.json: 100%|##########| 17.2MB / 17.2MB            

  ...adapter_model.safetensors:   0%|          |  557kB /  168MB            


✅ Success! Your LoRA adapters have been pushed to the Hugging Face Hub.
You can view and use your model from this link:
https://huggingface.co/Black4cosmos/Llama-3-Engineering-Finetune-QLoRA
